This notebook is the Data Extraction and Risk Cohort Definition engine. Its single goal is to take a raw list of industry codes and transform them into a small, clean set of statistically stable Risk Cohorts based on how businesses actually default over a specified window, at least within the SBA loan portfolio.

This notebook will generate a transitory database, cotemporally partitioned for a training/testing split.  It will also be subdivided into four (default) risk cohorts, each with its own collective default curve

Establish path variables, including temporarily amending the sys path such that it can find the utils folder regardless of where the notebook exists in the warehouse environment.

In [1]:
import sqlite3
import numpy as np
import pandas as pd
import os
import sys
import logging
from pathlib import Path
from datetime import datetime

# 1. Logging Infrastructure Configuration for Regulatory Audit Compliance
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(filename)s:%(lineno)d | %(message)s',
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler('notebook_1_warehouse_data_prep.log', mode='w')
    ]
)
logger = logging.getLogger("Notebook_1_Warehouse_Prep")

# 2. Dynamically locate data warehouse ROOT directory
notebook_path = Path(os.getcwd())
root_dir = notebook_path
while root_dir.name != "data_warehouse" and root_dir.parent != root_dir:
    root_dir = root_dir.parent
logger.info(f"Data warehouse root anchor verified at: {root_dir}")

# 3. Add the ROOT path to Python's system path for imports
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))
logger.info("Root directory successfully inserted into sys.path for local utilities.")

# 4. Bring in your custom daemon to manage database structural locations
from utils.geography_daemon import GeographyDaemon

# 5. Establish Explicit Data Type Engine Dictionary for Schema Enforcement
# This prevents pandas from casting zip codes to floats or leaving trailing spaces
DATA_TYPES = {
    'asofdate': str,
    'naicscode': str,
    'loanstatus': str,
    'borrzip': str,
    'projectcounty': str,
    'projectstate': str
}




2026-07-01 09:05:17,719 | INFO | 261386999.py:26 | Data warehouse root anchor verified at: /Users/bonwier/PythonProjects/data_warehouse
2026-07-01 09:05:17,720 | INFO | 261386999.py:31 | Root directory successfully inserted into sys.path for local utilities.


#### Cell 2:

Define duration of hazard curves, and split between short and long duration loans.  This split is based upon operational understanding of sba loan origination and observations from prior models. It is recommended that it remain at 60 months.  Same for Minimum Loan Threshold, as lower numbers guarantee data sparcity issues and artifacts in the statistical results.

In [2]:
# ==============================================================================
# CORE SURVIVAL ENGINE HYPERPARAMETERS
# ==============================================================================
# The maximum observation window for survival curves (5 Years / 60 Months).
# Right-censoring will be explicitly enforced past this operational threshold in RAM.
WINDOW_MONTHS = 60

# The structural threshold separating working capital/liquidity notes from long-term debt assets.
TERM_SPLIT_MONTHS = 60

# Log hyperparameter definitions for structural reproducibility and model tracking
logger.info("--- Warehouse Structural Hyperparameters Loaded ---")
logger.info(f"Explicit Right-Censoring Horizon (WINDOW_MONTHS): {WINDOW_MONTHS} months")
logger.info(f"Asset Class Duration Partition (TERM_SPLIT_MONTHS): {TERM_SPLIT_MONTHS} months")


2026-07-01 09:05:35,822 | INFO | 989154655.py:12 | --- Warehouse Structural Hyperparameters Loaded ---
2026-07-01 09:05:35,823 | INFO | 989154655.py:13 | Explicit Right-Censoring Horizon (WINDOW_MONTHS): 60 months
2026-07-01 09:05:35,824 | INFO | 989154655.py:14 | Asset Class Duration Partition (TERM_SPLIT_MONTHS): 60 months


#### Cell 3:

Define 4-digit industries to be ovserved, and classified.

In [3]:
# Defined target group of ~25 4-digit NAICS codes for Peri-Urban Agriculture
target_naics_cohort = np.array([
    1111, # Oilseed and Grain Farming
    1112, # Vegetable and Melon Farming
    1113, # Fruit and Tree Nut Farming
    1114, # Greenhouse, Nursery, and Floriculture Production
    1119, # Other Crop Farming
    1121, # Cattle Ranching and Farming
    1122, # Hog and Pig Farming
    1123, # Poultry and Egg Production
    1124, # Sheep and Goat Farming
    1125, # Aquaculture
    1129, # Other Animal Production
    1151, # Support Activities for Crop Production
    1152, # Support Activities for Animal Production
    3111, # Animal Food Manufacturing
    3112, # Grain and Oilseed Milling
    3113, # Sugar and Confectionery Product Manufacturing
    3114, # Fruit and Vegetable Preserving and Specialty Food
    3115, # Dairy Product Manufacturing
    3116, # Animal Slaughtering and Processing
    3118, # Bakeries and Tortilla Manufacturing
    3119, # Other Food Manufacturing
    3121, # Beverage Manufacturing (e.g., localized cideries/wineries)
    4244, # Grocery and Related Product Merchant Wholesalers
    4245, # Farm Product Raw Material Merchant Wholesalers
    4452  # Specialty Food Retailers (e.g., local farm stands/markets)
])

# For robust relational querying, create a string-cast version of the targets
# This prevents database string vs numeric comparison bugs during the filtration step
target_naics_strings = [str(code) for code in target_naics_cohort]

logger.info(f"Target industry boundary locked. Vectorized {len(target_naics_cohort)} sectors.")
print(f"Template initialized for {len(target_naics_cohort)} target industry sectors.")



2026-07-01 09:05:41,373 | INFO | 206733456.py:34 | Target industry boundary locked. Vectorized 25 sectors.
Template initialized for 25 target industry sectors.


#### Cell 4:

Connect to primary (sba 7a) database and class for temporal and spatial normalization of data.

In [4]:
# Initialize the daemon to resolve paths natively
daemon = GeographyDaemon()

# Construct the absolute path to the core sba analytical database
db_path = root_dir / "databases" / "sba_7a_analysis.db"

# Connect directly to the 7(a) analytical database
conn = sqlite3.connect(db_path)

# --- PERFORMANCE ENHANCEMENT PRAGMAS ---
# Optimize the database connection for high-velocity vectorized reads
conn.execute("PRAGMA journal_mode=WAL;")
conn.execute("PRAGMA synchronous=NORMAL;")

logger.info(f"Established optimized WAL connection to master analytical warehouse at: {db_path.name}")
print(f"Connected to analytical warehouse at: {db_path.name}")


2026-07-01 09:05:47,142 | INFO | 764957485.py:15 | Established optimized WAL connection to master analytical warehouse at: sba_7a_analysis.db
Connected to analytical warehouse at: sba_7a_analysis.db


#### Cell  5:

Extract loans from defined industries and load dataframe into memory.  This cell also normalizes naics codes against historical revisions, also truncating them to the 4-digit codes used for the model.  It cleans the resulting data, and also normalizes spatial data to compensate for changes in zip codes and fips codes over the years, mapping each loan to its appropriate county by current fips definitions.

In [5]:
# =====================================================================
# Cell 5: Streamlined High-Speed Parent-Hierarchy Extraction
# =====================================================================

# 1. Derive the clean 3-digit parent prefixes from curated target strings
parent_3d_codes = sorted(list(set(code[:3] for code in target_naics_strings)))

# Construct highly optimized, direct text equals constraints for a perfect B-Tree index hit
sql_index_clauses = " OR ".join([f"naics_3d = '{prefix}'" for prefix in parent_3d_codes])
logger.info(f"Targeting indexed parent sector macro regions in warehouse: {parent_3d_codes}")

# 2. Optimized Database Extraction
sba_conn = sqlite3.connect(db_path)
sba_conn.execute("PRAGMA journal_mode=WAL;")
sba_conn.execute("PRAGMA synchronous=NORMAL;")

# Query pulls only pristine, pre-engineered records matching our target 3-digit universes
query = f"""
SELECT 
    naics_4d, 
    naics_3d, 
    terminmonths, 
    survival_months_raw, 
    isdefaulted, 
    loanstatus, 
    asofdate, 
    approvaldate, 
    firstdisbursementdate, 
    paidinfulldate, 
    chargeoffdate, 
    grosschargeoffamount, 
    borrzip, 
    standardized_fips
FROM model_cohort_2003_present
WHERE {sql_index_clauses};
"""

raw_loans_df = pd.read_sql_query(query, sba_conn)
sba_conn.close()

# 3. Vectorized Structural Liquidity Term Split
# Establish short-term vs long-term structural partitions in memory
# If term is greater than or equal to 60 months, it is stamped as a long-duration asset (1)
raw_loans_df['is_long_duration'] = np.where(raw_loans_df['terminmonths'] >= TERM_SPLIT_MONTHS, 1, 0)

logger.info(f"🎉 Success! High-velocity parent extraction complete.")
logger.info(f" • In-Memory Dataset Footprint: {raw_loans_df.shape[0]:,} loans successfully pulled.")
print(f"Loaded {len(raw_loans_df):,} pre-cleaned, pre-indexed parent population loans from database.")




2026-07-01 09:05:54,208 | INFO | 3483807849.py:10 | Targeting indexed parent sector macro regions in warehouse: ['111', '112', '115', '311', '312', '424', '445']
2026-07-01 09:06:00,621 | INFO | 3483807849.py:46 | 🎉 Success! High-velocity parent extraction complete.
2026-07-01 09:06:00,622 | INFO | 3483807849.py:47 |  • In-Memory Dataset Footprint: 91,169 loans successfully pulled.
Loaded 91,169 pre-cleaned, pre-indexed parent population loans from database.


#### Cell 6:

Normalize datetime fields for pandas analysis.

In [6]:
# Convert all target date columns into proper datetime formats
date_cols = ['approvaldate', 'firstdisbursementdate', 'paidinfulldate', 'chargeoffdate']
for col in date_cols:
    raw_loans_df[col] = pd.to_datetime(raw_loans_df[col], errors='coerce')

# --- DATA QUALITY CHECKPOINT & AUDIT LOGGING ---
logger.info("--- Portfolio Temporal Profile Audit ---")
for col in date_cols:
    missing_count = raw_loans_df[col].isna().sum()
    logger.info(f" • Column '{col}': {missing_count:,} missing/unparseable values.")

# Extract operational time horizons to verify portfolio boundaries
min_app = raw_loans_df['approvaldate'].min()
max_app = raw_loans_df['approvaldate'].max()
if pd.notna(min_app) and pd.notna(max_app):
    logger.info(f"Pristine chronological envelope spanning from {min_app.strftime('%Y-%m-%d')} to {max_app.strftime('%Y-%m-%d')}.")

print("Date fields normalized into high-precision datetime formats.")



2026-07-01 09:06:08,231 | INFO | 1226642643.py:7 | --- Portfolio Temporal Profile Audit ---
2026-07-01 09:06:08,233 | INFO | 1226642643.py:10 |  • Column 'approvaldate': 0 missing/unparseable values.
2026-07-01 09:06:08,234 | INFO | 1226642643.py:10 |  • Column 'firstdisbursementdate': 0 missing/unparseable values.
2026-07-01 09:06:08,236 | INFO | 1226642643.py:10 |  • Column 'paidinfulldate': 28,554 missing/unparseable values.
2026-07-01 09:06:08,237 | INFO | 1226642643.py:10 |  • Column 'chargeoffdate': 80,416 missing/unparseable values.
2026-07-01 09:06:08,241 | INFO | 1226642643.py:16 | Pristine chronological envelope spanning from 2002-10-02 to 2026-03-25.
Date fields normalized into high-precision datetime formats.


#### Cell 7: Engineering the Duration (t) and Event (d) Columns

This cell contains the core survival logic. For each loan, its tracking timeline begins at its firstdisbursementdate.

If a loan defaulted (status matches CHG_OFF or has a charge-off date), its timeline ends at the chargeoffdate and event = 1.

If a loan paid in full, its timeline ends at the paidinfulldate and event = 0.

If it is still active, its timeline ends at the dataset's tracking boundary (2026-03-31 based on your raw files) and event = 0.

In [7]:
# =====================================================================
# Cell 7: In-Memory Timeline Engineering & 5-Year Right-Censoring (Fixed)
# =====================================================================

# 1. Dynamically identify the data boundary based on the latest available record date
DATASET_END_BOUND = pd.to_datetime(raw_loans_df['asofdate']).max()
print(f"Dynamic Right-Censoring Boundary established at: {DATASET_END_BOUND.strftime('%Y-%m-%d')}")

# 2. Project-Specific Temporal Engineering & Right-Censoring in RAM
# Ensure chargeoffdate is evaluated cleanly (handles datetime objects, strings, or NaT)
has_valid_chargeoff = raw_loans_df['chargeoffdate'].notna() & (raw_loans_df['chargeoffdate'] != '')

# New True Default Condition: Triggers on upstream flag OR presence of an explicit chargeoff date
is_true_default = (raw_loans_df['isdefaulted'] == 1) | has_valid_chargeoff

# Enforce your 5-year (60-month) tracking constraints dynamically on the raw timelines
raw_loans_df['event_occurred'] = np.where(
    is_true_default & (raw_loans_df['survival_months_raw'] <= WINDOW_MONTHS),
    1,
    0
)

raw_loans_df['survival_months'] = np.minimum(raw_loans_df['survival_months_raw'], WINDOW_MONTHS)

# --- PORTFOLIO RISK DIAGNOSTIC AUDIT ---
logger.info("--- Baseline Survival Metric Generation Complete ---")
observed_events = raw_loans_df['event_occurred'].sum()
censored_count = len(raw_loans_df) - observed_events

# Cross-check exactly how many revised PIF anomalies were caught and fixed in memory
pif_anomalies_caught = raw_loans_df[(raw_loans_df['loanstatus'] == 'P I F') & (has_valid_chargeoff)]

logger.info(f" • Total Portfolio Profile: {len(raw_loans_df):,} loans analyzed.")
logger.info(f" • Default Events Observed (Event=1): {observed_events:,} ({ (observed_events/len(raw_loans_df))*100 :.2f}%)")
logger.info(f" • Right-Censored Profiles (Event=0): {censored_count:,} ({ (censored_count/len(raw_loans_df))*100 :.2f}%)")
logger.info(f" • Revised PIF Mismatch Remediation: {len(pif_anomalies_caught):,} recovery anomalies forced to Event=1.")
print(f"Timeline tracking engineered for {len(raw_loans_df):,} funded loans.")




Dynamic Right-Censoring Boundary established at: 2026-03-31
2026-07-01 09:06:22,218 | INFO | 1896472250.py:26 | --- Baseline Survival Metric Generation Complete ---
2026-07-01 09:06:22,230 | INFO | 1896472250.py:33 |  • Total Portfolio Profile: 91,169 loans analyzed.
2026-07-01 09:06:22,231 | INFO | 1896472250.py:34 |  • Default Events Observed (Event=1): 6,726 (7.38%)
2026-07-01 09:06:22,232 | INFO | 1896472250.py:35 |  • Right-Censored Profiles (Event=0): 84,443 (92.62%)
2026-07-01 09:06:22,233 | INFO | 1896472250.py:36 |  • Revised PIF Mismatch Remediation: 1,095 recovery anomalies forced to Event=1.
Timeline tracking engineered for 91,169 funded loans.


####  Cell 8: The Dual-Duration Structural Split

Finally, we apply your operational rule: split each industry into two completely different groups based on a 60-month threshold.

In [8]:
# Display an in-memory cross-tabulation of your structured data grid
summary_grid = pd.crosstab(raw_loans_df['naics_4d'], raw_loans_df['is_long_duration'])
summary_grid.columns = ['Short-Term (<60mo)', 'Long-Term (>=60mo)']

# --- CREDIT EVENT RISK CROSS-TABULATION ENHANCEMENT ---
# Build an audit profile to map how actual defaults (Event=1) track across these cells
event_grid = pd.crosstab(
    index=raw_loans_df['naics_4d'],
    columns=raw_loans_df['is_long_duration'],
    values=raw_loans_df['event_occurred'],
    aggfunc='sum'
).fillna(0).astype(int)
event_grid.columns = ['Short-Term Defaults', 'Long-Term Defaults']

# Merge the structural footprint matrix with observed defaults for a complete risk overview
comprehensive_audit_grid = pd.concat([summary_grid, event_grid], axis=1)

logger.info("--- Portfolio Term Structure Risk Segmentation Matrix ---")
logger.info(f"\n{comprehensive_audit_grid.to_string()}")

# Render the primary summary grid to match your original notebook visual interface
comprehensive_audit_grid




2026-07-01 09:06:32,515 | INFO | 1350811575.py:18 | --- Portfolio Term Structure Risk Segmentation Matrix ---
2026-07-01 09:06:32,519 | INFO | 1350811575.py:19 | 
          Short-Term (<60mo)  Long-Term (>=60mo)  Short-Term Defaults  Long-Term Defaults
naics_4d                                                                                 
1111                      58                 294                    2                   5
1112                      27                 144                    7                   6
1113                      26                 212                    0                   4
1114                      69                 604                   16                  14
1119                      65                 382                    6                   0
1121                     218                 732                   12                   3
1122                      12                  69                    0                   0
1123                     62

,Short-Term (<60mo),Long-Term (>=60mo),Short-Term Defaults,Long-Term Defaults
naics_4d,,,,
1111,58,294,2,5
1112,27,144,7,6
1113,26,212,0,4
1114,69,604,16,14
1119,65,382,6,0
1121,218,732,12,3
1122,12,69,0,0
1123,620,5212,24,42
1124,3,23,0,1


#### Cell 9: Define the Vector Sampling Function
To cluster curves, we need them to be the exact same shape and length. This cell creates a helper function that takes a raw, irregular survival history and turns it into a standardized 10-point vector (one point for each year out to year 10).

In [9]:
def extract_standard_survival_vector(group_df, max_months=60):
    """
    Computes an exact Kaplan-Meier cumulative default curve, properly 
    accounting for right-censoring at the exact month of exit.
    """
    annual_checkpoints = np.arange(12, max_months + 1, 12)
    num_years = len(annual_checkpoints)
    
    if len(group_df) == 0:
        return np.zeros(num_years)
        
    # Aggregate total defaults (d_i) and total exits (defaults + censored c_i) at each month
    # This forms the exact basis of true actuarial life-table logic
    monthly_stats = group_df.groupby('survival_months').agg(
        d_i=('event_occurred', 'sum'),
        total_exits=('survival_months', 'count')
    ).sort_index()
    
    # Calculate the rolling risk pool (n_i) dynamically
    # Start with total population, then subtract the cumulative sum of previous exits
    total_records = len(group_df)
    cumulative_exits_prior = monthly_stats['total_exits'].cumsum().shift(1).fillna(0)
    monthly_stats['n_i'] = total_records - cumulative_exits_prior
    
    # Calculate the Kaplan-Meier step multiplier: (1 - d_i / n_i)
    # Handle edge case where risk pool drops to zero to avoid divide-by-zero errors
    monthly_stats['step_survival'] = np.where(
        monthly_stats['n_i'] > 0,
        1.0 - (monthly_stats['d_i'] / monthly_stats['n_i']),
        1.0
    )
    
    monthly_stats['cumulative_survival'] = monthly_stats['step_survival'].cumprod()
    
    # Map back to your standardized annual checkpoints
    standardized_default_vector = []
    for month in annual_checkpoints:
        historical_steps = monthly_stats[monthly_stats.index <= month]
        if len(historical_steps) > 0:
            latest_survival = historical_steps['cumulative_survival'].iloc[-1]
        else:
            latest_survival = 1.0
        
        # Append cumulative default probability (1 - Survival)
        standardized_default_vector.append(1.0 - latest_survival)
        
    return np.array(standardized_default_vector)


logger.info("Warehouse-horizon Kaplan-Meier survival engine compiled successfully.")


2026-07-01 09:06:52,109 | INFO | 1291818327.py:50 | Warehouse-horizon Kaplan-Meier survival engine compiled successfully.


#### Cell 10: Run the Population Loop & Filter Thin Cells

This cell runs our matrix builder across all possible combinations in the database. It uses the MIN_LOAN_THRESHOLD parameter we defined in Step 1 to protect the model from running on noisy, low-volume data cells.

In [10]:
# Initialize empty lists to store our processed results and metadata
curve_matrix = []
metadata_records = []

# Enforce final target alignment strings from your original target list
target_naics_strings = [str(int(code)).zfill(4) for code in target_naics_cohort]
logger.info("--- Beginning In-Memory Kaplan-Meier Profile Compilation with Parent Backfills ---")

# Dynamically compute the size of your annual timeline vector (60 months / 12 = 5 points)
num_annual_checkpoints = WINDOW_MONTHS // 12

# Iterate strictly through the intended target codes and binary duration structures
for naics in target_naics_strings:
    for is_long in [0, 1]:
        segment_label = f"{naics}_Short" if is_long == 0 else f"{naics}_Long"
        
        # Filter raw portfolio data for the specific 4-digit industry segment
        group_data = raw_loans_df[(raw_loans_df['naics_4d'] == naics) & (raw_loans_df['is_long_duration'] == is_long)].copy()
        loan_count = len(group_data)
        
        # -----------------------------------------------------------------
        # DROP-FREE STRATEGY ENGINE (Zero-Volume Protection)
        # -----------------------------------------------------------------
        if loan_count == 0:
            # Isolate the 3-digit parent prefix (e.g., '1122' -> '112')
            parent_prefix = naics[:3]
            
            # Pull the full parent population matching this duration split from memory
            parent_data = raw_loans_df[(raw_loans_df['naics_3d'] == parent_prefix) & (raw_loans_df['is_long_duration'] == is_long)].copy()
            parent_loan_count = len(parent_data)
            
            if parent_loan_count > 0:
                logger.warning(
                    f"[STRATEGY OVERRIDE: PARENT_HIERARCHY] - Segment {segment_label} is completely empty "
                    f"at 4-digit level. Routing to parent '{parent_prefix}' ({parent_loan_count} loans)."
                )
                active_data = parent_data
                strategy_applied = "PARENT_HIERARCHY"
                audit_count = parent_loan_count
            else:
                logger.error(
                    f"[STRATEGY OVERRIDE: PENALTY_BOX] - Both target {segment_label} and parent '{parent_prefix}' "
                    f"possess zero records. Placing segment in Penalty Box with a flat curve."
                )
                active_data = pd.DataFrame()
                strategy_applied = "PENALTY_BOX"
                audit_count = 0
        else:
            # Robust or native data cell: track normally
            active_data = group_data
            strategy_applied = "NATIVE_CELL"
            audit_count = loan_count
            
        # -----------------------------------------------------------------
        # SURVIVAL MARK COMPILATION (Leveraging Pristine Cell 7 Tensors)
        # -----------------------------------------------------------------
        if len(active_data) > 0:
            # Generate the standardized default vector via the registered Cell 9 engine
            default_curve = extract_standard_survival_vector(active_data, max_months=WINDOW_MONTHS)
            effective_defaults = active_data['event_occurred'].sum()
        else:
            # Dynamically match your project's target timeline array width
            default_curve = np.zeros(num_annual_checkpoints)
            effective_defaults = 0
            
        # -----------------------------------------------------------------
        # CACHE COMPILED MATRIX & METADATA
        # -----------------------------------------------------------------
        curve_matrix.append(default_curve)
        metadata_records.append({
            'naics_4d': naics,
            'is_long_duration': is_long,
            'total_loans': loan_count,
            'total_defaults': effective_defaults if strategy_applied == "NATIVE_CELL" else 0,
            'label': segment_label,
            'manifesto_strategy': strategy_applied,
            'effective_sample_size': audit_count,
            'effective_defaults': effective_defaults
        })

# Convert our list of vectors into a solid numpy array for downstream modeling
X_curves = np.array(curve_matrix)
df_curves_metadata = pd.DataFrame(metadata_records)

logger.info(f"Curve Matrix compiled successfully. Shape: {X_curves.shape}")
print(f"\nSuccessfully generated {X_curves.shape} standardized {num_annual_checkpoints}-year default curves.")
print("\n--- Strategy Application Summary ---")
print(df_curves_metadata['manifesto_strategy'].value_counts())




2026-07-01 09:07:03,178 | INFO | 4205197735.py:7 | --- Beginning In-Memory Kaplan-Meier Profile Compilation with Parent Backfills ---
2026-07-01 09:07:03,941 | INFO | 4205197735.py:85 | Curve Matrix compiled successfully. Shape: (50, 5)

Successfully generated (50, 5) standardized 5-year default curves.

--- Strategy Application Summary ---
manifesto_strategy
NATIVE_CELL    50
Name: count, dtype: int64


#### Cell 11: Inspecting the Raw Curve Grid

Before throwing these curves into an unsupervised clustering algorithm, let's look at a quick snapshot of the raw data array to make sure the values are mathematically sound.

In [11]:
# =====================================================================
# Cell 11: Dynamic Cumulative Default Vector Diagnostics Check
# =====================================================================

# Derive the exact number of years present in the feature matrix dynamically
num_years_compiled = X_curves.shape[1]

# Wrap our numpy matrix in a DataFrame using the dynamic horizon bounds
curve_columns = [f"Year_{i}" for i in range(1, num_years_compiled + 1)]
diagnostics_df = pd.DataFrame(X_curves, columns=curve_columns)

# Attach descriptors to know exactly which row is which
diagnostics_df.insert(0, 'Segment', [r['label'] for r in metadata_records])
diagnostics_df.insert(1, 'Total_Loans', [r['total_loans'] for r in metadata_records])

# --- MONOTONICITY VIOLATION PROTECTION ENGINE ---
# Automatically verify that cumulative default values never decrease over time
diffs = diagnostics_df[curve_columns].diff(axis=1).iloc[:, 1:]
violations = (diffs < 0).sum().sum()

if violations > 0:
    logger.error(f"CRITICAL SURVIVAL BUG: Found {violations} monotonicity drops in cumulative curves!")
else:
    logger.info(f"Pass: Cumulative curves verified as perfectly monotonic across all {num_years_compiled} annual tracking marks.")

# Look at the first 5 records to verify they start low and scale up smoothly
diagnostics_df.head()



2026-07-01 09:07:19,195 | INFO | 3134125166.py:24 | Pass: Cumulative curves verified as perfectly monotonic across all 5 annual tracking marks.


,Segment,Total_Loans,Year_1,Year_2,Year_3,Year_4,Year_5
0,1111_Short,58,0.0,0.000000,0.000000,0.160839,0.160839
1,1111_Long,294,0.0,0.003690,0.013571,0.024444,0.024444
2,1112_Short,27,0.0,0.000000,0.176471,0.326203,0.508690
3,1112_Long,144,0.0,0.017041,0.037379,0.037379,0.067409
4,1113_Short,26,0.0,0.000000,0.000000,0.000000,0.000000


In [12]:
# =====================================================================
# Cell 12: 4-Digit Industry Baseline Risk Profile Leaderboard
# =====================================================================

logger.info("Compiling raw industry risk leaderboards directly from compiled vectors...")

# 1. Reconstruct a clean diagnostic frame directly from our metadata records
df_industry_profiles = pd.DataFrame(metadata_records)

# 2. Map the 5-Year (60-month) maximum cumulative default rate from X_curves
# The last column in X_curves represents Year_5 (Month 60) cumulative default probability
df_industry_profiles['raw_60mo_default_rate'] = X_curves[:, -1]

# 3. Calculate a simple baseline risk ranking sorted by volume and default rate
df_leaderboard = df_industry_profiles[[
    'naics_4d', 'is_long_duration', 'total_loans', 'total_defaults', 'raw_60mo_default_rate'
]].copy()

df_leaderboard = df_leaderboard.sort_values(by=['is_long_duration', 'raw_60mo_default_rate'], ascending=[True, False])

# 4. Print validation table for upcoming Notebook 2 modeling
print("\n=== UNBIASED WAREHOUSE PROFILE LEADERBOARD (60-MONTH HORIZON) ===")
print(df_leaderboard.to_string(index=False, formatters={
    'raw_60mo_default_rate': '{:.2%}'.format
}))

logger.info("Baseline profile leaderboard successfully established. Premature linkages bypassed.")




2026-07-01 09:07:51,224 | INFO | 833096339.py:5 | Compiling raw industry risk leaderboards directly from compiled vectors...

=== UNBIASED WAREHOUSE PROFILE LEADERBOARD (60-MONTH HORIZON) ===
naics_4d  is_long_duration  total_loans  total_defaults raw_60mo_default_rate
    1129                 0           41              13                62.85%
    1152                 0           93              27                61.54%
    3113                 0           81              25                57.77%
    3119                 0          309             104                56.96%
    4244                 0         1331             447                55.93%
    4452                 0         1446             604                53.60%
    3112                 0           21               5                53.25%
    1112                 0           27               7                50.87%
    3114                 0           85              27                48.16%
    3118                 0  

In [13]:
# =====================================================================
# Cell 13: Industry Description Mapping & Baseline PD Extraction
# =====================================================================
import pandas as pd
import json

logger.info("Extracting pristine baseline 5-year default rates from curve matrices...")

# 1. EXTRACT TRUE 5-YEAR CUMULATIVE PD VIA THE VECTOR MATRIX
# Since curves are capped at 5 years (60 months), the very last index (-1) holds the exact 5-year PD
df_curves_metadata['baseline_5yr_pd'] = [round(curve[-1] * 100, 2) for curve in curve_matrix]

# 2. GLOBAL JSON REFERENCE HANDSHAKE
# Pull in industry descriptions to keep the warehouse human-readable
universal_json_path = root_dir / "databases" / "naics_4d.json"
try:
    with open(universal_json_path, 'r', encoding='utf-8') as f:
        universal_dictionary = json.load(f)
    
    # Map descriptions seamlessly using your clean 4-digit text strings
    df_curves_metadata['sector_name'] = df_curves_metadata['naics_4d'].astype(str).str.strip().map(universal_dictionary)
    logger.info("Universal global JSON reference handshake complete.")
except Exception as e:
    logger.error(f"Failed to ingest universal NAICS JSON reference schema: {e}")
    df_curves_metadata['sector_name'] = "Curated Target Sector"

df_curves_metadata['sector_name'] = df_curves_metadata['sector_name'].fillna("Unknown Economy Sector")

# 3. Order the un-biased metadata dataframe logically by duration and raw risk profiles
df_curves_metadata = df_curves_metadata.sort_values(
    by=['is_long_duration', 'baseline_5yr_pd'],
    ascending=[True, False]
).reset_index(drop=True)

print(f"\n🎉 Successfully mapped {len(df_curves_metadata)} industry duration cells.")
print("\n--- Complete Unbiased 5-Year Industry Baseline Grid ---")

# Expand display options so pandas layouts look beautiful in your console view
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', 45)

print(df_curves_metadata[['label', 'sector_name', 'total_loans', 'total_defaults', 'baseline_5yr_pd']])


2026-07-01 09:08:16,609 | INFO | 3934332221.py:7 | Extracting pristine baseline 5-year default rates from curve matrices...
2026-07-01 09:08:16,615 | INFO | 3934332221.py:22 | Universal global JSON reference handshake complete.

🎉 Successfully mapped 50 industry duration cells.

--- Complete Unbiased 5-Year Industry Baseline Grid ---
         label                                   sector_name  total_loans  total_defaults  baseline_5yr_pd
0   1129_Short                       Other Animal Production           41              13            62.85
1   1152_Short      Support Activities for Animal Production           93              27            61.54
2   3113_Short  Sugar and Confectionery Product Manufactu...           81              25            57.77
3   3119_Short                      Other Food Manufacturing          309             104            56.96
4   4244_Short  Grocery and Related Product Merchant Whol...         1331             447            55.93
5   4452_Short        

In [14]:
# =====================================================================
# Cell 14: The Sparse Handling Manifesto Identification Dashboard
# =====================================================================

# 1. Establish your project's target portfolio slice
target_naics_strings = [str(int(code)).zfill(4) for code in target_naics_cohort]
target_portfolio_df = raw_loans_df[raw_loans_df['naics_4d'].isin(target_naics_strings)].copy()

# Construct uniform segment labels to audit data density at the asset level
target_portfolio_df['segment_label'] = np.where(
    target_portfolio_df['is_long_duration'] == 0, 
    target_portfolio_df['naics_4d'] + "_Short", 
    target_portfolio_df['naics_4d'] + "_Long"
)

# 2. Scan the portfolio segments to flag sparse data cells
sparse_exceptions = []
unique_segments = sorted(target_portfolio_df['segment_label'].unique())

# Define your strict sparse threshold for the model training footprint
MIN_TRAIN_LOAN_THRESHOLD = 20

for label in unique_segments:
    seg_slice = target_portfolio_df[target_portfolio_df['segment_label'] == label]
    total_loans = len(seg_slice)
    
    # Pre-calculate project-specific 5-year default counts for the audit display
    defaults_5yr = ((seg_slice['isdefaulted'] == 1) & (seg_slice['survival_months_raw'] <= WINDOW_MONTHS)).sum()
    
    if total_loans < MIN_TRAIN_LOAN_THRESHOLD:
        sparse_exceptions.append({
            'label': label,
            'total_loans': total_loans,
            'total_defaults': defaults_5yr
        })

# 3. Initialize the dynamic treatment dictionary dashboard
# The keys will be the unique string descriptors ('1124_Short', '1125_Short', etc.)
sparse_treatment_manifesto = {}

if len(sparse_exceptions) > 0:
    print("🚨 SPARSE SEGMENTS IDENTIFIED OVER TARGET PORTFOLIO 🚨")
    print("--------------------------------------------------")
    for item in sparse_exceptions:
        # Set the strict risk-averse default stance: STRIP entirely from the final engine
        sparse_treatment_manifesto[item['label']] = 'STRIP'
        print(f" • Segment: {item['label']:11} | Target Loans: {item['total_loans']:4} | Defaults: {item['total_defaults']:3} | Action: 'STRIP'")
        
    print("\n💡 ACTION REQUIRED: Review the dictionary above. To apply custom adjustments, ")
    print("modify individual keys in the cell below using the following options:")
    print(" --> 'STRIP'            : Drops these rows completely to insulate the credit engine.")
    print(" --> 'PENALTY_BOX'      : Force-assigns the segment to the absolute riskiest clustered cohort.")
    print(" --> 'PARENT_HIERARCHY' : Forces the segment to inherit the hazard curve of its 3-digit parent.")
else:
    print("🎉 Pass: No sparse industry-duration segments detected across your 5-year portfolio horizon.")


🚨 SPARSE SEGMENTS IDENTIFIED OVER TARGET PORTFOLIO 🚨
--------------------------------------------------
 • Segment: 1122_Short  | Target Loans:   12 | Defaults:   0 | Action: 'STRIP'
 • Segment: 1124_Short  | Target Loans:    3 | Defaults:   0 | Action: 'STRIP'
 • Segment: 1125_Short  | Target Loans:   11 | Defaults:   1 | Action: 'STRIP'

💡 ACTION REQUIRED: Review the dictionary above. To apply custom adjustments, 
modify individual keys in the cell below using the following options:
 --> 'STRIP'            : Drops these rows completely to insulate the credit engine.
 --> 'PENALTY_BOX'      : Force-assigns the segment to the absolute riskiest clustered cohort.
 --> 'PARENT_HIERARCHY' : Forces the segment to inherit the hazard curve of its 3-digit parent.


In [15]:
# =====================================================================
# Cell 15: Automated Structural Flagging & Master Data Serialization
# =====================================================================
import os
import sqlite3
import pandas as pd
import numpy as np

logger.info("Initializing project-neutral data layer serialization pipeline...")

# 1. Standardize Target Codes as Zero-Padded Strings
target_naics_strings = [str(int(code)).zfill(4) for code in target_naics_cohort]

# 2. Capture ALL Extracted Records (Core + Proxy Background Rows)
# We no longer drop rows here; we preserve the entire hierarchy on disk
df_warehouse = raw_loans_df.copy()

# 3. FIELD 1: Identify the Definitive Core Target Sample vs. 3D Background Proxies
df_warehouse['is_core_sample'] = np.where(df_warehouse['naics_4d'].isin(target_naics_strings), 1, 0)

# 4. FIELD 2: Vectorized Automated Sparsity Flagging (N < 20)
# We calculate data density strictly using the core target sample space
cell_counts = df_warehouse[df_warehouse['is_core_sample'] == 1].groupby(['naics_4d', 'is_long_duration']).size().reset_index(name='cell_total_loans')
df_warehouse = df_warehouse.merge(cell_counts, on=['naics_4d', 'is_long_duration'], how='left')

MIN_TRAIN_LOAN_THRESHOLD = 20
df_warehouse['is_sparse_cell'] = np.where(
    (df_warehouse['is_core_sample'] == 1) & (df_warehouse['cell_total_loans'] < MIN_TRAIN_LOAN_THRESHOLD), 
    1, 
    0
)
df_warehouse = df_warehouse.drop(columns=['cell_total_loans'])

# 5. FIELD 3: Calculate the 3-Digit Parent Neighborhood Footprint Size
parent_counts = df_warehouse.groupby(['naics_3d', 'is_long_duration']).size().reset_index(name='parent_3d_count')
df_warehouse = df_warehouse.merge(parent_counts, on=['naics_3d', 'is_long_duration'], how='left')

# 6. Compile the Comprehensive Audit Log Table for Notebook 2 Visibility
df_audit_log = df_warehouse.groupby(['naics_4d', 'naics_3d', 'is_long_duration', 'is_core_sample', 'is_sparse_cell', 'parent_3d_count']).agg(
    total_loans=('event_occurred', 'count'),
    total_defaults=('event_occurred', 'sum')
).reset_index()

# 7. Execute High-Velocity Serialization to Disk
transitory_dir = root_dir / "databases" / "transitory"
transitory_dir.mkdir(parents=True, exist_ok=True)
db_write_path = transitory_dir / "peri_urban_ag_analysis.db"

logger.info(f"Writing un-biased micro-data and audit logs to: {db_write_path}")
conn = sqlite3.connect(db_write_path)
try:
    conn.execute("PRAGMA journal_mode=WAL;")
    conn.execute("PRAGMA synchronous=OFF;") # Maximize bulk writing velocity
    
    # Table 1: Pristine, individual loan-level microdata modeling layer
    df_warehouse.to_sql(
        name="source_loans_snapshot",
        con=conn,
        if_exists="replace",
        index=False,
        chunksize=10000
    )
    
    # Table 2: Sparsity reference matrix for automated Notebook 2 methodology routing
    df_audit_log.to_sql(
        name="audit_sparse_treatments_log",
        con=conn,
        if_exists="replace",
        index=False
    )
    
    logger.info("🎉 Database serialization complete. Both tables written cleanly with structural proxy tracking.")
finally:
    conn.close()

# 8. Print Definitive Execution Report
print("\n=== CORESYNC WAREHOUSE COMPILATION SUMMARY ===")
print(f" • Total Portfolio Pool on Disk: {len(df_warehouse):,}")
print(f"   --> Core Target Loans:        {df_warehouse['is_core_sample'].sum():,}")
print(f"   --> Background 3D Proxies:    {(df_warehouse['is_core_sample'] == 0).sum():,}")
print(f" • Active Sparse Target Cells:   {df_warehouse['is_sparse_cell'].sum():,}")
print(f" • Target Database Location:     databases/transitory/peri_urban_ag_analysis.db")


2026-07-01 09:13:42,966 | INFO | 3555169570.py:9 | Initializing project-neutral data layer serialization pipeline...
2026-07-01 09:13:43,206 | INFO | 3555169570.py:49 | Writing un-biased micro-data and audit logs to: /Users/bonwier/PythonProjects/data_warehouse/databases/transitory/peri_urban_ag_analysis.db
2026-07-01 09:13:44,068 | INFO | 3555169570.py:72 | 🎉 Database serialization complete. Both tables written cleanly with structural proxy tracking.

=== CORESYNC WAREHOUSE COMPILATION SUMMARY ===
 • Total Portfolio Pool on Disk: 91,169
   --> Core Target Loans:        45,651
   --> Background 3D Proxies:    45,518
 • Active Sparse Target Cells:   26
 • Target Database Location:     databases/transitory/peri_urban_ag_analysis.db
